## CUDA GEMM Benchmark (Colab)

This notebook assumes you copied the repo's `cuda-benchmark/` folder to the root of your Google Drive at:

`MyDrive/cuda-benchmark/`

It will:
- Mount Google Drive
- Compile `cublas_benchmark.cu` with `nvcc`
- Run the PyTorch benchmark
- Run the cuBLAS benchmark
- Run the plotting script (writes plots to `results/plots/`)


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# (Optional) sanity checks
!nvidia-smi
!nvcc --version


In [ ]:
# Go to cuda-benchmark folder in Drive
import os
bench_dir = '/content/drive/MyDrive/cuda-benchmark'
assert os.path.isdir(bench_dir), f"Missing {bench_dir}. Copy the repo's cuda-benchmark folder there."
%cd {bench_dir}
!ls -la


In [ ]:
# Ensure plotting deps exist (PyTorch is typically preinstalled on Colab GPUs)
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'available', torch.cuda.is_available())"
!pip -q install matplotlib


In [ ]:
# Create output directories
!mkdir -p results results/plots


In [ ]:
# Compile cuBLAS benchmark
# Note: Colab has CUDA + cuBLAS available on the system image
!nvcc -O3 --std=c++17 cublas_benchmark.cu -o cublas_benchmark -lcublas
!ls -la


In [ ]:
# Run PyTorch benchmark (writes results/pytorch_benchmark.json)
!python pytorch_benchmark.py \
  --sizes 512,1024,2048,4096 \
  --batch_size 256 \
  --in_features 4096 \
  --out_features 4096 \
  --warmup_iters 10 \
  --timed_iters 50 \
  --output results/pytorch_benchmark.json


In [ ]:
# Run cuBLAS benchmark (writes results/cublas_benchmark.csv)
!./cublas_benchmark \
  --sizes 512,1024,2048,4096 \
  --batch_size 256 \
  --in_features 4096 \
  --out_features 4096 \
  --warmup_iters 10 \
  --timed_iters 50 \
  --output results/cublas_benchmark.csv


In [ ]:
# Plot comparison curves (writes PNGs into results/plots/)
!python plot_benchmarks.py \
  --pytorch_json results/pytorch_benchmark.json \
  --cublas_csv results/cublas_benchmark.csv \
  --out_dir results/plots
!ls -la results/plots
